<a href="https://colab.research.google.com/github/Manjot-Coder/IP_LabAssignments/blob/main/Assignment-6/Q1_Q2_Q3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Q1,Q2,Q3


from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io

# Step 1: Upload image
uploaded = files.upload()
filename = next(iter(uploaded))

img = Image.open(io.BytesIO(uploaded[filename])).convert("L")
image = np.array(img, dtype=float)

# Step 2: Take filter size
size = int(input("Enter filter size (odd number, e.g. 3 or 5): "))

if size <= 0 or size % 2 == 0:
    raise ValueError("Filter size must be a positive odd number.")

# Step 3: Select padding method
print("\nPadding Methods:")
print("1. Zero Padding")
print("2. Replication Padding")
print("3. Reflect Padding")

choice = int(input("Enter padding choice: "))

padding_modes = {
    1: "constant",
    2: "edge",
    3: "reflect"
}

if choice not in padding_modes:
    raise ValueError("Invalid padding choice.")

mode = padding_modes[choice]

# Step 4: Create padding
pad = size // 2
padded = np.pad(image, pad, mode=mode)

# Step 5: Create filter masks

# 1. Averaging Filter
average_filter = np.ones((size, size), dtype=float)
average_filter = average_filter / (size * size)

# 2. Weighted Filter: h(x,y) = max(|x|, |y|)
center = size // 2
weighted_filter = np.zeros((size, size), dtype=float)

for i in range(size):
    for j in range(size):
        x = i - center
        y = j - center
        weighted_filter[i, j] = max(abs(x), abs(y))

# Normalize weighted filter
if np.sum(weighted_filter) == 0:
    weighted_filter[center, center] = 1

weighted_filter = weighted_filter / np.sum(weighted_filter)

# 3. Gaussian Filter
sigma = float(input("\nEnter sigma for Gaussian filter: "))

if sigma <= 0:
    raise ValueError("Sigma must be positive.")

gaussian_filter = np.zeros((size, size), dtype=float)

for i in range(size):
    for j in range(size):
        x = i - center
        y = j - center

        gaussian_filter[i, j] = np.exp(
            -(x*x + y*y) / (2 * sigma*sigma)
        )

# Normalize Gaussian filter
gaussian_filter = gaussian_filter / np.sum(gaussian_filter)


# Step 6: Apply filter using convolution
def apply_filter(padded_image, kernel, original_shape):

    h, w = original_shape
    k = kernel.shape[0]
    result = np.zeros((h, w), dtype=float)

    for i in range(h):
        for j in range(w):

            region = padded_image[i:i+k, j:j+k]

            result[i, j] = np.sum(region * kernel)

    return np.clip(result, 0, 255).astype(np.uint8)


# Step 7: Apply all three filters
average_result = apply_filter(padded, average_filter, image.shape)
weighted_result = apply_filter(padded, weighted_filter, image.shape)
gaussian_result = apply_filter(padded, gaussian_filter, image.shape)


# Step 8: Display results
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.imshow(image, cmap="gray")
plt.title("Original Image")
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(average_result, cmap="gray")
plt.title("Averaging Filter")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(weighted_result, cmap="gray")
plt.title("Weighted Filter")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.imshow(gaussian_result, cmap="gray")
plt.title("Gaussian Filter")
plt.axis("off")

plt.tight_layout()
plt.show()


# Step 9: Save output images
Image.fromarray(average_result).save("averaging_filter.jpg")
Image.fromarray(weighted_result).save("weighted_filter.jpg")
Image.fromarray(gaussian_result).save("gaussian_filter.jpg")

print("All three filtered images saved successfully!")

